# NLP Lab 04
Ali Zaib | FA23-BAI-004 | BAI7

Run cells in order from this extracted folder. Tables and plots are saved in `results`.

In [ ]:
%pip install -q -r requirements.txt

In [1]:
import re, json, html, time, random
from pathlib import Path
from collections import Counter
from functools import partial
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nltk, spacy
from nltk.stem import PorterStemmer, SnowballStemmer
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from rapidfuzz.distance import Levenshtein

DATA, OUT = Path("data"), Path("results")
OUT.mkdir(exist_ok=True)
from nltk.corpus import stopwords
nltk.data.path.insert(0, str(DATA / "nltk_data"))
stop_words = set(stopwords.words("english"))
porter, snowball = PorterStemmer(), SnowballStemmer("english")
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    from spacy.cli import download
    download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")
except OverflowError:
    nlp = spacy.load("en_core_web_sm", exclude=["tokenizer"])

def tokens(text):
    return re.findall(r"[A-Za-z]+(?:'[A-Za-z]+)?", text)

def save_table(frame, filename):
    display(frame)
    frame.to_csv(OUT / filename, index=False, encoding="utf-8-sig")

## Task 1 Lowercasing

In [2]:
corpus = json.loads((DATA / "case_corpus.json").read_text())
lowered = [text.lower() for text in corpus]
save_table(pd.DataFrame({"Original": corpus, "Lowercase": lowered}), "case_comparison.csv")
before_vocab = {word for text in corpus for word in tokens(text)}
after_vocab = {word for text in lowered for word in tokens(text)}
print("Vocabulary before:", len(before_vocab))
print("Vocabulary after:", len(after_vocab))
save_table(pd.DataFrame([
    ["US / us", "United States / object pronoun", "us"],
    ["Apple / apple", "Company / fruit", "apple"],
    ["May / may", "Month or name / modal verb", "may"]
], columns=["Forms", "Original meanings", "Merged form"]), "case_ambiguity.csv")

Vocabulary before: 47
Vocabulary after: 40


,Original,Lowercase
0,Apple launched a new phone.,apple launched a new phone.
1,I ate an apple today.,i ate an apple today.
2,APPLE juice tastes sweet.,apple juice tastes sweet.
3,The US signed an agreement.,the us signed an agreement.
4,Please send us the report.,please send us the report.
5,US researchers contacted us.,us researchers contacted us.
6,May is a warm month.,may is a warm month.
7,You may join the class.,you may join the class.
8,I use Python for NLP.,i use python for nlp.
9,The python moved slowly.,the python moved slowly.


,Forms,Original meanings,Merged form
0,US / us,United States / object pronoun,us
1,Apple / apple,Company / fruit,apple
2,May / may,Month or name / modal verb,may


Lowercasing reduces duplicate vocabulary entries and helps case-insensitive matching. It can harm NER because names and acronyms lose their capitalization clues, so US becomes indistinguishable from us. Sentiment analysis may lose emphasis such as GREAT versus great. Search can lose entity distinctions such as Apple the company versus apple the fruit, although case-insensitive search often improves recall. Keeping original text alongside a normalized version preserves useful information.

## Task 2 Noise removal

In [3]:
def remove_urls(text):
    return re.sub(r"https?://\S+|www\.\S+", " ", text)

def remove_html(text):
    return html.unescape(re.sub(r"<[^>]+>", " ", text))

def handle_mentions_hashtags(text):
    text = re.sub(r"(?<!\w)@\w+", " ", text)
    return re.sub(r"#(\w+)", r"\1", text)

def remove_emojis(text):
    return re.sub("[\U0001F000-\U0001FAFF\u2600-\u27BF\uFE0F\u200D]", " ", text)

def remove_punctuation(text):
    return re.sub(r"[^\w\s]", " ", text)

def remove_extra_whitespace(text):
    return re.sub(r"\s+", " ", text).strip()

noise_steps = [remove_urls, remove_html, handle_mentions_hashtags,
               remove_emojis, remove_punctuation, remove_extra_whitespace]

def clean_text(text, steps=None):
    for step in noise_steps if steps is None else steps:
        text = step(text)
    return text

noisy = pd.read_csv(DATA / "noisy_reviews.csv")
noisy["cleaned"] = noisy.text.map(clean_text)
print("Entries:", len(noisy))
save_table(noisy.head(5), "noise_examples.csv")
noisy.to_csv(OUT / "all_cleaned_reviews.csv", index=False)
wrong_order = [remove_punctuation, remove_urls, remove_html,
               handle_mentions_hashtags, remove_emojis, remove_extra_whitespace]
probe = '<b>Learn C++</b> at https://code.org @ali #coding for $100'
print("Correct order:", clean_text(probe))
print("Punctuation first:", clean_text(probe, wrong_order))
print("Order changes output:", clean_text(probe) != clean_text(probe, wrong_order))

Entries: 17
Correct order: Learn C at coding for 100
Punctuation first: b Learn C b at https code org ali coding for 100
Order changes output: True


,text,cleaned
0,I love C++! Visit https://code.org @ali #coding 😊,I love C Visit coding
1,<b>Great</b> phone for $100! #deal 😍,Great phone for 100 deal
2,Not good!!! @shop https://bad.example 😠,Not good
3,Email me at user@example.com <i>please</i>.,Email me at user example com please
4,Battery lasts 10-hours... #battery 🔋,Battery lasts 10 hours battery


URLs, HTML and mentions should be handled before punctuation: deleting punctuation first destroys their recognizable syntax. The five-example table shows over-cleaning: C++ becomes C, $100 loses its currency marker, and an email loses its address structure. Apostrophe removal also turns don't into don t. A production pipeline should protect technical terms, currency, email addresses and negation before deleting punctuation; removing emojis can also remove sentiment cues.

## Task 3 Stop words

In [4]:
stopword_list = stopwords.words("english")
print("First 20 NLTK English stop words:", stopword_list[:20])
print("Total:", len(stopword_list))
reviews = pd.read_csv(DATA / "movie_reviews.csv")
review_sentences = json.loads((DATA / "review_sentences.json").read_text())
sentence_tokens = [tokens(sentence.lower()) for sentence in review_sentences]
filtered_sentences = [[word for word in row if word not in stop_words] for row in sentence_tokens]
all_before = [word for row in sentence_tokens for word in row]
all_after = [word for row in filtered_sentences for word in row]
stats = pd.DataFrame([
    ["Before", len(all_before), len(set(all_before)), np.mean([len(row) for row in sentence_tokens])],
    ["After", len(all_after), len(set(all_after)), np.mean([len(row) for row in filtered_sentences])]
], columns=["Version", "Tokens", "Vocabulary", "Average sentence length"])
save_table(stats, "stopword_statistics.csv")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, words, title in zip(axes, [all_before, all_after], ["Before removal", "After removal"]):
    top = Counter(words).most_common(15)
    ax.bar([word for word, count in top], [count for word, count in top])
    ax.set_title(title); ax.set_ylabel("Frequency"); ax.tick_params(axis="x", rotation=60)
fig.tight_layout(); fig.savefig(OUT / "top15_stopwords.png", dpi=180)
display(fig)
plt.close(fig)
choices = [
    ("movie", "add", "Common review-domain label; little topic discrimination."),
    ("film", "add", "Repeated medium name rather than an opinion."),
    ("movies", "add", "Plural of the common review-domain label."),
    ("films", "add", "Plural medium name; often redundant."),
    ("cinema", "add", "Broad domain label in this movie-only collection."),
    ("not", "remove", "Preserve explicit negation."),
    ("no", "remove", "Preserve absence and negation."),
    ("nor", "remove", "Preserve coordinated negation."),
    ("but", "remove", "Preserve contrast between opinions."),
    ("against", "remove", "Preserve opposition or negative stance.")]
custom_stops = set(stop_words)
for word, action, reason in choices:
    if action == "add": custom_stops.add(word)
    else: custom_stops.discard(word)
save_table(pd.DataFrame(choices, columns=["Word", "Change", "Justification"]), "custom_stopwords.csv")
print("Actual changes:", len(custom_stops.symmetric_difference(stop_words)))
print("Domain stop words are task-dependent; deleting film/movie may harm topic classification.")

First 20 NLTK English stop words: ['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been']
Total: 198
Actual changes: 10
Domain stop words are task-dependent; deleting film/movie may harm topic classification.


,Version,Tokens,Vocabulary,Average sentence length
0,Before,64275,9154,19.146559
1,After,33898,9010,10.097706


<Figure size 1400x500 with 2 Axes>

,Word,Change,Justification
0,movie,add,Common review-domain label; little topic discr...
1,film,add,Repeated medium name rather than an opinion.
2,movies,add,Plural of the common review-domain label.
3,films,add,Plural medium name; often redundant.
4,cinema,add,Broad domain label in this movie-only collection.
5,not,remove,Preserve explicit negation.
6,no,remove,Preserve absence and negation.
7,nor,remove,Preserve coordinated negation.
8,but,remove,Preserve contrast between opinions.
9,against,remove,Preserve opposition or negative stance.


## Task 4 Risks of stop word removal

In [5]:
negation_examples = ["The movie is not good.", "I am never happy with this service.",
                     "There is no problem.", "I do not recommend this movie.", "The ending is not bad."]

def remove_stops(text, stopset=stop_words):
    return " ".join(word for word in tokens(text.lower()) if word not in stopset)

safe_stops = stop_words - {"no", "not", "nor", "never", "neither", "without", "against", "but", "don't", "isn't", "wasn't", "can't", "won't"}
risks = ["not good → good: negative becomes positive", "never remains: no negation flip here",
         "no problem → problem: reassuring meaning is lost", "not recommend → recommend: rejection becomes endorsement",
         "not bad → bad: mild praise becomes negative"]
sentiment = pd.DataFrame({"Original": negation_examples,
                          "Blind removal": [remove_stops(s) for s in negation_examples],
                          "Safe removal": [remove_stops(s, safe_stops) for s in negation_examples],
                          "Manual sentiment observation": risks})
save_table(sentiment, "negation_risks.csv")
translations = pd.read_csv(DATA / "urdu_translations.csv").fillna("")
save_table(translations, "translation_comparison.csv")
queries = ["How to not be charged?", "What is the status of my order?",
           "Can I cancel my order?", "Can I not cancel my order?", "Where is my order?"]
intent_notes = ["Loses not: avoiding a charge becomes charged.", "Keeps status/order; drops ownership and question wording.",
                "Keeps cancel/order; drops whether cancellation is possible.", "Collapses with the preceding query despite opposite intent.",
                "Drops where: delivery-location intent is weakened."]
save_table(pd.DataFrame({"Query": queries, "Reduced": [remove_stops(q) for q in queries],
                        "Risk": intent_notes}), "chatbot_intents.csv")
print("Sentiment observations are manual linguistic judgments, not a trained sentiment model.")

Sentiment observations are manual linguistic judgments, not a trained sentiment model.


,Original,Blind removal,Safe removal,Manual sentiment observation
0,The movie is not good.,movie good,movie not good,not good → good: negative becomes positive
1,I am never happy with this service.,never happy service,never happy service,never remains: no negation flip here
2,There is no problem.,problem,no problem,no problem → problem: reassuring meaning is lost
3,I do not recommend this movie.,recommend movie,not recommend movie,not recommend → recommend: rejection becomes e...
4,The ending is not bad.,ending bad,ending not bad,not bad → bad: mild praise becomes negative


,Original English,Reduced English,Original Urdu,Reduced Urdu,Original source,Reduced source,Meaning comparison
0,I do not want to cancel my order.,want cancel order,میں اپنا آرڈر منسوخ نہیں کرنا چاہتا ۔,آرڈر منسوخ کریں,MyMemory API retrieved 2026-10-25,MyMemory API retrieved 2026-10-25,Removing not reverses the cancellation prefere...
1,She is not going to school.,going school,Kon sa school ja raha hai,اسکول جانا,MyMemory API retrieved 2026-10-25,MyMemory API retrieved 2026-10-25,The original output is unreliable Roman Urdu (...
2,He gave the book to her.,gave book,اس نے کتاب اسے دی ۔,کتاب دی,MyMemory API retrieved 2026-10-25,MyMemory API retrieved 2026-10-25,"Removing he/to/her loses the giver, recipient ..."


,Query,Reduced,Risk
0,How to not be charged?,charged,Loses not: avoiding a charge becomes charged.
1,What is the status of my order?,status order,Keeps status/order; drops ownership and questi...
2,Can I cancel my order?,cancel order,Keeps cancel/order; drops whether cancellation...
3,Can I not cancel my order?,cancel order,Collapses with the preceding query despite opp...
4,Where is my order?,order,Drops where: delivery-location intent is weake...


Keep negation and contrast markers for sentiment, and preserve contractions as whole tokens. Negation scope still matters: a bag of words cannot fully represent not only good but excellent. For translation and chatbot intent detection, retaining the complete sentence is safer because pronouns, auxiliaries and question words carry meaning. The safe column re-tests all five sentiment examples. Translator outputs in the CSV include their source and a manual comparison; they are not an accuracy benchmark.

## Task 5 Porter stemming

In [6]:
words = ["running", "runs", "runner", "studies", "studying", "study", "happily", "happiness", "happy",
         "generalization", "generalize", "better", "was", "went", "feet", "children", "cars", "car", "played", "playing",
         "university", "universe", "universal", "organization", "organ", "organic", "news", "new", "relational", "relation",
         "generous", "generosity", "sky", "skies", "dying", "lying", "tying", "proceed", "proceeds", "proceeded"]
expected = {"running":"run", "runs":"run", "runner":"runner", "studies":"study", "studying":"study", "study":"study",
            "happily":"happy", "happiness":"happy", "happy":"happy", "generalization":"generalize", "generalize":"generalize",
            "better":"good", "was":"be", "went":"go", "feet":"foot", "children":"child", "cars":"car", "car":"car",
            "played":"play", "playing":"play", "university":"university", "universe":"universe", "universal":"universal",
            "organization":"organization", "organ":"organ", "organic":"organic", "news":"news", "new":"new",
            "relational":"relational", "relation":"relation", "generous":"generous", "generosity":"generosity",
            "sky":"sky", "skies":"sky", "dying":"die", "lying":"lie", "tying":"tie", "proceed":"proceed",
            "proceeds":"proceed", "proceeded":"proceed"}
stems = pd.DataFrame([{"Word": word, "Stem": porter.stem(word), "Reference form": expected[word],
                       "Correct/Incorrect": "Correct" if porter.stem(word)==expected[word] else "Incorrect",
                       "Snowball": snowball.stem(word)} for word in words])
save_table(stems, "stemming_table.csv")
print("Correct/Incorrect means exact agreement with the stated reference form, not universal retrieval quality.")
print("A non-word stem is permitted by stemming, so this strict lexical criterion is intentionally limited.")
pairs = [("Over", "university", "universe"), ("Over", "organization", "organ"), ("Over", "communism", "community"),
         ("Under", "run", "runner"), ("Under", "happy", "happily"), ("Under", "good", "better")]
errors = pd.DataFrame([[kind,a,b,porter.stem(a),porter.stem(b)] for kind,a,b in pairs],
                     columns=["Type", "Word 1", "Word 2", "Stem 1", "Stem 2"])
save_table(errors, "stemming_errors.csv")
save_table(stems[stems.Stem != stems.Snowball][["Word", "Stem", "Snowball"]], "porter_snowball_differences.csv")

Correct/Incorrect means exact agreement with the stated reference form, not universal retrieval quality.
A non-word stem is permitted by stemming, so this strict lexical criterion is intentionally limited.


,Word,Stem,Reference form,Correct/Incorrect,Snowball
0,running,run,run,Correct,run
1,runs,run,run,Correct,run
2,runner,runner,runner,Correct,runner
3,studies,studi,study,Incorrect,studi
4,studying,studi,study,Incorrect,studi
5,study,studi,study,Incorrect,studi
6,happily,happili,happy,Incorrect,happili
7,happiness,happi,happy,Incorrect,happi
8,happy,happi,happy,Incorrect,happi
9,generalization,gener,generalize,Incorrect,general


,Type,Word 1,Word 2,Stem 1,Stem 2
0,Over,university,universe,univers,univers
1,Over,organization,organ,organ,organ
2,Over,communism,community,commun,commun
3,Under,run,runner,run,runner
4,Under,happy,happily,happi,happili
5,Under,good,better,good,better


,Word,Stem,Snowball
9,generalization,gener,general
10,generalize,gener,general
12,was,wa,was
30,generous,gener,generous
38,proceeds,proce,proceed


## Task 6 Lemmatization versus stemming

In [7]:
paragraph = (DATA / "lemmatization_paragraph.txt").read_text()
print("Paragraph word count:", len(tokens(paragraph)))
doc = nlp(paragraph)
comparison = pd.DataFrame([[t.text, t.pos_, porter.stem(t.text.lower()), t.lemma_] for t in doc if t.is_alpha],
                          columns=["Token", "POS", "Stem", "Lemma"])
save_table(comparison, "stem_lemma_comparison.csv")
display(comparison[comparison.Token.str.lower().isin(["better", "was", "children", "went", "feet"])])
repeats = 30
start = time.perf_counter()
for _ in range(repeats):
    [porter.stem(t.text.lower()) for t in doc if t.is_alpha]
stem_time = (time.perf_counter()-start)/repeats
start = time.perf_counter()
for _ in range(repeats):
    [(t.lemma_, t.pos_) for t in nlp(paragraph) if t.is_alpha]
lemma_time = (time.perf_counter()-start)/repeats
save_table(pd.DataFrame({"Method":["Porter on pre-tokenized words", "spaCy full trained pipeline"],
                         "Seconds per paragraph":[stem_time,lemma_time]}), "timing.csv")

Paragraph word count: 119


,Token,POS,Stem,Lemma
0,The,DET,the,the
1,children,NOUN,children,child
2,were,AUX,were,be
3,running,VERB,run,run
4,through,ADP,through,through
...,...,...,...,...
114,together,ADV,togeth,together
115,during,ADP,dure,during
116,the,DET,the,the
117,following,ADJ,follow,following


,Token,POS,Stem,Lemma
1,children,NOUN,children,child
12,was,AUX,wa,be
31,better,ADJ,better,well
41,went,VERB,went,go
48,feet,NOUN,feet,foot
58,was,AUX,wa,be


,Method,Seconds per paragraph
0,Porter on pre-tokenized words,0.001147
1,spaCy full trained pipeline,0.024998


Lemmatization uses grammatical context and POS to choose dictionary forms: was is an auxiliary mapped to be, and comparative better can map to good when tagged as an adjective. Better can also be an adverb, so the observed POS and lemma must be checked rather than assumed. Porter only applies word-form rules and cannot generally recover irregular forms. Choose stemming for lightweight indexing and approximate matching; choose lemmatization when readable base forms and grammatical distinctions matter. Timing excludes model loading; the spaCy measurement includes tokenization and all trained components, while Porter receives existing tokens, so this compares practical workloads rather than isolated equivalent operations.

## Task 7 Configurable preprocessing and classification

In [8]:
class Preprocessor:
    def __init__(self, lowercase=True, noise=False, stopwords=False, mode=None):
        if mode not in {None, "stem", "lemma"}:
            raise ValueError("mode must be None, stem or lemma")
        self.lowercase, self.noise = lowercase, noise
        self.stopwords, self.mode = stopwords, mode

    def __call__(self, text):
        if self.lowercase: text = text.lower()
        if self.noise: text = clean_text(text)
        if self.mode == "lemma":
            doc = [t for t in nlp(text) if t.is_alpha]
            if self.stopwords: doc = [t for t in doc if t.text.lower() not in safe_stops]
            return [t.lemma_ for t in doc]
        else:
            words = tokens(text)
        if self.stopwords: words = [w for w in words if w.lower() not in safe_stops]
        if self.mode == "stem": words = [porter.stem(w) for w in words]
        return words

configs = {"Lowercase": Preprocessor(),
           "Lowercase + stopwords": Preprocessor(stopwords=True),
           "Lowercase + Porter": Preprocessor(mode="stem"),
           "Lowercase + lemma": Preprocessor(mode="lemma")}
train_idx, test_idx = train_test_split(np.arange(len(reviews)), test_size=0.25,
                                      random_state=42, stratify=reviews.label)
rows = []
for name, processor in configs.items():
    processed = [processor(text) for text in reviews.text]
    vectorizer = TfidfVectorizer(tokenizer=str.split, preprocessor=None, token_pattern=None, lowercase=False)
    strings = [" ".join(row) for row in processed]
    X_train = vectorizer.fit_transform([strings[i] for i in train_idx])
    X_test = vectorizer.transform([strings[i] for i in test_idx])
    model = LogisticRegression(max_iter=1000, random_state=42)
    model.fit(X_train, reviews.label.iloc[train_idx])
    predicted = model.predict(X_test)
    accuracy = accuracy_score(reviews.label.iloc[test_idx], predicted)
    rows.append([name,len({w for row in processed for w in row}),np.mean([len(row) for row in processed]),
                 len(vectorizer.vocabulary_),accuracy])
    (OUT / (name.replace(" ", "_").replace("+", "and") + "_classification.txt")).write_text(
        classification_report(reviews.label.iloc[test_idx],predicted,zero_division=0))
results = pd.DataFrame(rows,columns=["Configuration","Full corpus vocabulary","Average tokens per document",
                                    "Training TF-IDF vocabulary","Test accuracy"])
save_table(results,"classification_results.csv")
best = results[results["Test accuracy"] == results["Test accuracy"].max()]
analysis = [f"The dataset contains {len(reviews)} actual movie reviews with balanced positive and negative labels.",
            f"All configurations use the same {len(train_idx)} training and {len(test_idx)} test reviews (seed 42).",
            "TF-IDF is fitted on training reviews only; the full-corpus vocabulary column is descriptive.",
            "Best observed configuration(s): " + ", ".join(best.Configuration) + f"; accuracy {best['Test accuracy'].iloc[0]:.1%}.",
            "Stemming merges forms and stop-word removal reduces tokens; lemma uses spaCy tokenization, so these scores compare full pipelines.",
            "This small fixed split is exploratory: close scores do not establish a reliable winner without more data and cross-validation."]
print("\n".join(analysis))
(OUT / "classification_analysis.txt").write_text("\n".join(analysis))

The dataset contains 100 actual movie reviews with balanced positive and negative labels.
All configurations use the same 75 training and 25 test reviews (seed 42).
TF-IDF is fitted on training reviews only; the full-corpus vocabulary column is descriptive.
Best observed configuration(s): Lowercase + stopwords; accuracy 64.0%.
Stemming merges forms and stop-word removal reduces tokens; lemma uses spaCy tokenization, so these scores compare full pipelines.
This small fixed split is exploratory: close scores do not establish a reliable winner without more data and cross-validation.
Out[0]: 586


,Configuration,Full corpus vocabulary,Average tokens per document,Training TF-IDF vocabulary,Test accuracy
0,Lowercase,9432,629.24,8065,0.48
1,Lowercase + stopwords,9277,350.33,7912,0.64
2,Lowercase + Porter,7118,629.24,6156,0.52
3,Lowercase + lemma,7515,628.35,6478,0.52


## Task 8 Minimum edit distance

In [9]:
def levenshtein_recursive(a, b):
    if not a: return len(b)
    if not b: return len(a)
    return min(levenshtein_recursive(a[:-1], b) + 1,
               levenshtein_recursive(a, b[:-1]) + 1,
               levenshtein_recursive(a[:-1], b[:-1]) + (a[-1] != b[-1]))

print("Recursive cat → cut:", levenshtein_recursive("cat", "cut"))
print("Without memoization, overlapping subproblems cause exponential growth; use short inputs only.")

def edit_table(a, b, substitution=1):
    dp = [[0]*(len(b)+1) for _ in range(len(a)+1)]
    for i in range(len(a)+1): dp[i][0] = i
    for j in range(len(b)+1): dp[0][j] = j
    for i in range(1,len(a)+1):
        for j in range(1,len(b)+1):
            dp[i][j] = min(dp[i-1][j]+1, dp[i][j-1]+1,
                           dp[i-1][j-1]+(0 if a[i-1]==b[j-1] else substitution))
    return dp

def backtrace(a,b,dp,substitution=1):
    i,j = len(a),len(b)
    operations = []
    while i or j:
        if i and j and a[i-1]==b[j-1] and dp[i][j]==dp[i-1][j-1]:
            operations.append(("match",a[i-1],b[j-1],0)); i-=1; j-=1
        elif i and j and dp[i][j]==dp[i-1][j-1]+substitution:
            operations.append(("substitute",a[i-1],b[j-1],substitution)); i-=1; j-=1
        elif i and dp[i][j]==dp[i-1][j]+1:
            operations.append(("delete",a[i-1],"",1)); i-=1
        else:
            operations.append(("insert","",b[j-1],1)); j-=1
    return list(reversed(operations))

pairs = [("intention","execution"),("kitten","sitting"),("sunday","saturday")]
distance_rows = []
for a,b in pairs:
    for cost in [1,2]:
        dp = edit_table(a,b,cost)
        table = pd.DataFrame(dp,index=["∅"]+list(a),columns=["∅"]+list(b))
        print(f"{a} → {b}, substitution cost={cost}, distance={dp[-1][-1]}")
        display(table)
        table.to_csv(OUT/f"dp_{a}_{b}_cost{cost}.csv")
        operations = backtrace(a,b,dp,cost)
        save_table(pd.DataFrame(operations,columns=["Operation","Source","Target","Cost"]),f"operations_{a}_{b}_cost{cost}.csv")
        assert sum(op[3] for op in operations)==dp[-1][-1]
        assert "".join(op[1] for op in operations)==a
        assert "".join(op[2] for op in operations)==b
        distance_rows.append([a,b,cost,dp[-1][-1]])
save_table(pd.DataFrame(distance_rows,columns=["Source","Target","Substitution cost","Distance"]),"distance_comparison.csv")

Recursive cat → cut: 1
Without memoization, overlapping subproblems cause exponential growth; use short inputs only.
intention → execution, substitution cost=1, distance=5
intention → execution, substitution cost=2, distance=8
kitten → sitting, substitution cost=1, distance=3
kitten → sitting, substitution cost=2, distance=5
sunday → saturday, substitution cost=1, distance=3
sunday → saturday, substitution cost=2, distance=4


,∅,e,x,e,c,u,t,i,o,n
∅,0,1,2,3,4,5,6,7,8,9
i,1,1,2,3,4,5,6,6,7,8
n,2,2,2,3,4,5,6,7,7,7
t,3,3,3,3,4,5,5,6,7,8
e,4,3,4,3,4,5,6,6,7,8
n,5,4,4,4,4,5,6,7,7,7
t,6,5,5,5,5,5,5,6,7,8
i,7,6,6,6,6,6,6,5,6,7
o,8,7,7,7,7,7,7,6,5,6
n,9,8,8,8,8,8,8,7,6,5


,Operation,Source,Target,Cost
0,substitute,i,e,1
1,substitute,n,x,1
2,substitute,t,e,1
3,substitute,e,c,1
4,substitute,n,u,1
5,match,t,t,0
6,match,i,i,0
7,match,o,o,0
8,match,n,n,0


,∅,e,x,e,c,u,t,i,o,n
∅,0,1,2,3,4,5,6,7,8,9
i,1,2,3,4,5,6,7,6,7,8
n,2,3,4,5,6,7,8,7,8,7
t,3,4,5,6,7,8,7,8,9,8
e,4,3,4,5,6,7,8,9,10,9
n,5,4,5,6,7,8,9,10,11,10
t,6,5,6,7,8,9,8,9,10,11
i,7,6,7,8,9,10,9,8,9,10
o,8,7,8,9,10,11,10,9,8,9
n,9,8,9,10,11,12,11,10,9,8


,Operation,Source,Target,Cost
0,delete,i,,1
1,substitute,n,e,2
2,substitute,t,x,2
3,match,e,e,0
4,insert,,c,1
5,substitute,n,u,2
6,match,t,t,0
7,match,i,i,0
8,match,o,o,0
9,match,n,n,0


...

,∅,s,a,t,u,r,d,a,y
∅,0,1,2,3,4,5,6,7,8
s,1,0,1,2,3,4,5,6,7
u,2,1,1,2,2,3,4,5,6
n,3,2,2,2,3,3,4,5,6
d,4,3,3,3,3,4,3,4,5
a,5,4,3,4,4,4,4,3,4
y,6,5,4,4,5,5,5,4,3


,Operation,Source,Target,Cost
0,match,s,s,0
1,insert,,a,1
2,insert,,t,1
3,match,u,u,0
4,substitute,n,r,1
5,match,d,d,0
6,match,a,a,0
7,match,y,y,0


,∅,s,a,t,u,r,d,a,y
∅,0,1,2,3,4,5,6,7,8
s,1,0,1,2,3,4,5,6,7
u,2,1,2,3,2,3,4,5,6
n,3,2,3,4,3,4,5,6,7
d,4,3,4,5,4,5,4,5,6
a,5,4,3,4,5,6,5,4,5
y,6,5,4,5,6,7,6,5,4


,Operation,Source,Target,Cost
0,match,s,s,0
1,insert,,a,1
2,insert,,t,1
3,match,u,u,0
4,substitute,n,r,2
5,match,d,d,0
6,match,a,a,0
7,match,y,y,0


,Source,Target,Substitution cost,Distance
0,intention,execution,1,5
1,intention,execution,2,8
2,kitten,sitting,1,3
3,kitten,sitting,2,5
4,sunday,saturday,1,3
5,sunday,saturday,2,4


Each DP cell is the minimum of deletion from above +1, insertion from the left +1, and diagonal match/substitution. Dynamic programming takes O(mn) time and O(mn) space here; backtrace follows one optimal alignment, with substitutions preferred on ties. With substitution cost 2, a substitution ties a delete-plus-insert, so optimal paths need not be unique. Open `Hand_DP_Worksheet.html`, print it, and fill the three blank grids by hand as required. Completed notebook tables are reference answers, not handwritten work.

## Task 9 Spelling correction

In [10]:
dictionary = pd.read_csv(DATA / "dictionary_frequency.csv", keep_default_na=False)
frequencies = dict(zip(dictionary.word,dictionary.frequency))
vocabulary = sorted(frequencies)
assert len(vocabulary) >= 5000
print("Dictionary words:",len(vocabulary))

def suggest(word,k=5,use_frequency=False):
    word = word.lower()
    ranked = [(candidate,Levenshtein.distance(word,candidate)) for candidate in vocabulary]
    ranked.sort(key=lambda row:(row[1],-frequencies[row[0]] if use_frequency else 0,row[0]))
    return ranked[:k]

misspellings = pd.read_csv(DATA / "misspellings.csv")
assert set(misspellings.correct).issubset(frequencies)
rows=[]
for typo,correct in misspellings.itertuples(index=False,name=None):
    baseline = suggest(typo)
    improved = suggest(typo,use_frequency=True)
    rows.append([typo,correct,baseline[0][0],improved[0][0],baseline[0][0]==correct,improved[0][0]==correct,
                 "; ".join(w for w,d in baseline),"; ".join(w for w,d in improved)])
corrections = pd.DataFrame(rows,columns=["Misspelling","Correct","Distance-only top1","Frequency top1","Baseline correct","Frequency correct","Baseline top5","Frequency top5"])
save_table(corrections,"spelling_results.csv")
a,b = corrections["Baseline correct"].mean(),corrections["Frequency correct"].mean()
print(f"Distance-only Top-1 accuracy: {a:.1%}")
print(f"Frequency tie-break Top-1 accuracy: {b:.1%}")
print(f"Change: {(b-a)*100:+.1f} percentage points on {len(corrections)} selected spelling errors.")
print("Distance-only ties use alphabetic order; frequency ties prefer larger Brown corpus counts.")
print("Frequency is only a prior: it can favor a common unintended word over a rare correct word.")
print("Both methods use the same fixed dictionary, distances and test list; this is a small illustrative benchmark.")
print("Real-word example:",suggest("their",use_frequency=True))
print("Their and there are valid words: distance zero favors the typed word, so context is needed to detect misuse.")
for x,y in [("kitten","sitting"),("recieve","receive"),("", "word")]:
    assert Levenshtein.distance(x,y)==edit_table(x,y)[-1][-1]
print("RapidFuzz uses the same unit-cost Levenshtein distance as Task 8, with faster ranking.")

Dictionary words: 40234
Distance-only Top-1 accuracy: 72.2%
Frequency tie-break Top-1 accuracy: 83.3%
Change: +11.1 percentage points on 18 selected spelling errors.
Distance-only ties use alphabetic order; frequency ties prefer larger Brown corpus counts.
Frequency is only a prior: it can favor a common unintended word over a rare correct word.
Both methods use the same fixed dictionary, distances and test list; this is a small illustrative benchmark.
Real-word example: [('their', 0), ('theirs', 1), ('heir', 1), ('ther', 1), ('thei', 1)]
Their and there are valid words: distance zero favors the typed word, so context is needed to detect misuse.
RapidFuzz uses the same unit-cost Levenshtein distance as Task 8, with faster ranking.


,Misspelling,Correct,Distance-only top1,Frequency top1,Baseline correct,Frequency correct,Baseline top5,Frequency top5
0,recieve,receive,relieve,relieve,False,False,relieve; believe; receave; recede; receive,relieve; believe; receive; relieved; recipe
1,definately,definitely,definitely,definitely,True,True,definitely; delicately; defiantly; definable; ...,definitely; delicately; definite; indefinitely...
2,acommodate,accommodate,accommodate,accommodate,True,True,accommodate; accommodated; accommodates; commo...,accommodate; accommodates; accommodated; commo...
3,seperate,separate,separate,separate,True,True,separate; desperate; generate; operate; separated,separate; operate; separated; desperate; generate
4,occured,occurred,occurred,occurred,True,True,occurred; accused; cured; obscured; occluded,occurred; occur; occupied; occurs; accused
5,untill,until,until,until,True,True,until; intial; still; till; untidy,until; still; till; untie; untold
6,enviroment,environment,environment,environment,True,True,environment; environments; enlistment; enrichm...,environment; environments; environmental; envi...
7,goverment,government,government,government,True,True,government; governments; movement; coherent; c...,government; movement; governments; movements; ...
8,becuase,because,became,because,False,True,became; because; accuse; base; beale,because; became; case; become; base
9,langauge,language,language,language,True,True,language; baggage; bandage; engage; gauge,language; languages; manage; engage; gauge
